#### Import libraries & set seeds

In [ ]:
# Standard Libaries
import sys
from pathlib import Path
import os
import random

# Setting Path
notebook_path = Path().resolve()
parent_dir = notebook_path.parent
sys.path.append(str(parent_dir))
sys.path.append(str(parent_dir / "src"))


# Third Party Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader

import pytorch_lightning as pl

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    mean_absolute_error,
    balanced_accuracy_score,
)


# Custom Libraries
from src.dataset import ImageDataset, load_image_dataframe
from src.transforms import get_transforms


# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pl.seed_everything(seed, workers=True, verbose=False)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(15)

#### Functions

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_labels, title="Confusion Matrix", normalize=True):
    if normalize:
        cm = confusion_matrix(y_true, y_pred, normalize="true")
    else:
        cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Predicted", fontsize=14)
    ax.set_ylabel("True", fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    plt.tight_layout()
    return fig


def create_predictions(probs):
    argmax = np.argmax(probs, axis=1)
    number_of_classes = probs.shape[1]
    class_indices = np.arange(number_of_classes)
    expected_value = np.dot(probs, class_indices)

    return argmax, expected_value


def plot_expected_value_distributions(task_results, task_name="Unknown", save_path=None, save_name=None):
    class_names = {0: "Poor", 1: "Adequate", 2: "Good"}
    colors = {0: "red", 1: "orange", 2: "green"}

    evs = np.array(task_results["evs"])
    labels = np.array(task_results["labels"])

    plt.figure(figsize=(6, 4))
    for c in [0, 1, 2]:
        mask = labels == c
        if np.any(mask):
            sns.kdeplot(
                evs[mask],
                label=class_names[c],
                color=colors[c],
                fill=False,
                linewidth=3,
            )

    # X-axis from 0 to 2, ticks every 0.5
    plt.xlim(-0.5, 2.5)
    plt.xticks(np.arange(0.0, 2.1, 0.5), fontsize=12)
    y_max = plt.gca().get_ylim()[1]
    plt.yticks(np.arange(0, y_max, 0.5), fontsize=12)

    plt.xlabel("Expected Value", fontsize=14, fontweight="bold")
    plt.ylabel("Density", fontsize=14, fontweight="bold")
    plt.legend(fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    # Save figure if requested
    if save_path is not None:
        os.makedirs(save_path, exist_ok=True)
        if save_name is None:
            save_name = f"{task_name}_expected_value_distribution.png"
        save_full_path = os.path.join(save_path, save_name)
        plt.savefig(save_full_path, dpi=300, bbox_inches="tight")

    plt.show()


def apply_thresholds(values, thresholds):
    """Map expected values into discrete class predictions based on thresholds."""
    thresholds = sorted(thresholds)
    preds = np.zeros_like(values, dtype=int)
    for t in thresholds:
        preds += values > t
    return preds


def compute_task_metrics(labels, probs, evs, thresholds, task_name, n_classes):
    """
    Compute AUROC (macro), macro-MAE, macro Balanced Accuracy,
    and per-class Balanced Accuracy for a single task,
    ignoring labels == -100.
    """
    labels = np.array(labels)
    probs = np.array(probs)
    evs = np.array(evs)

    # mask out ignored labels
    mask = labels != -100
    labels, probs, evs = labels[mask], probs[mask], evs[mask]

    if len(labels) == 0:
        return {
            "auroc_macro": np.nan,
            "mae_macro": np.nan,
            "balanced_acc_macro": np.nan,
            "balanced_acc_per_class": [np.nan] * n_classes,
        }

    # AUROC (macro)
    try:
        if n_classes == 2:
            auroc = roc_auc_score(labels, probs[:, 1])
        else:
            auroc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        auroc = np.nan

    # Macro MAE
    maes = []
    for c in range(n_classes):
        mask_c = labels == c
        if np.any(mask_c):
            maes.append(mean_absolute_error(labels[mask_c], evs[mask_c]))
    mae_macro = float(np.mean(maes)) if maes else np.nan

    # Predictions
    preds = np.array([apply_thresholds(v, thresholds[task_name]) for v in evs])

    # Balanced Accuracy
    try:
        bal_acc_macro = balanced_accuracy_score(labels, preds)
        bal_acc_per_class = []
        for c in range(n_classes):
            mask_c = labels == c
            if np.any(mask_c):
                # One-vs-rest for each class
                bin_true = (labels == c).astype(int)
                bin_pred = (preds == c).astype(int)
                bal_acc_per_class.append(balanced_accuracy_score(bin_true, bin_pred))
            else:
                bal_acc_per_class.append(np.nan)
    except ValueError:
        bal_acc_macro = np.nan
        bal_acc_per_class = [np.nan] * n_classes

    return {
        "auroc_macro": auroc,
        "mae_macro": mae_macro,
        "balanced_acc_macro": bal_acc_macro,
        "balanced_acc_per_class": bal_acc_per_class,
    }


def compute_metrics(
    clean_labels,
    expansion_labels,
    oiq_labels,
    clean_probs,
    expansion_probs,
    oiq_probs,
    clean_evs,
    expansion_evs,
    oiq_evs,
    thresholds,
):
    """
    Compute metrics for clean, expansion, oiq tasks only.
    Retrograde is skipped.
    """
    metrics = {
        "clean": compute_task_metrics(clean_labels, clean_probs, clean_evs, thresholds, "clean", n_classes=3),
        "expansion": compute_task_metrics(
            expansion_labels, expansion_probs, expansion_evs, thresholds, "expansion", n_classes=3
        ),
        "oiq": compute_task_metrics(oiq_labels, oiq_probs, oiq_evs, thresholds, "oiq", n_classes=3),
    }
    return metrics


def evaluate_model(model, dataloader, thresholds):
    """Run model on dataloader and return per-task metrics."""
    model.eval()
    model.to("cuda")

    results = []

    with torch.no_grad():
        for batch in dataloader:
            images, mucosal_cleaning_label, expansion_label, oiq_label, retro_label = batch
            images = images.to("cuda")

            # Forward pass
            y_clean, y_expansion, y_oiq, y_retro = model(images)

            # Softmax probabilities
            clean_prob = torch.softmax(y_clean, dim=1)
            expansion_prob = torch.softmax(y_expansion, dim=1)
            oiq_prob = torch.softmax(y_oiq, dim=1)
            retro_prob = torch.softmax(y_retro, dim=1)  # not used

            # Expected values
            class_indices = torch.arange(3, device=oiq_prob.device).float()
            clean_ev = (clean_prob * class_indices).sum(dim=1).cpu()
            expansion_ev = (expansion_prob * class_indices).sum(dim=1).cpu()
            oiq_ev = (oiq_prob * class_indices).sum(dim=1).cpu()

            for i in range(images.size(0)):
                results.append(
                    {
                        "clean_label": mucosal_cleaning_label[i].item(),
                        "expansion_label": expansion_label[i].item(),
                        "oiq_label": oiq_label[i].item(),
                        "clean_prob": clean_prob[i].cpu().numpy(),
                        "expansion_prob": expansion_prob[i].cpu().numpy(),
                        "oiq_prob": oiq_prob[i].cpu().numpy(),
                        "clean_ev": clean_ev[i].item(),
                        "expansion_ev": expansion_ev[i].item(),
                        "oiq_ev": oiq_ev[i].item(),
                    }
                )

    # Convert to arrays
    clean_labels = np.array([r["clean_label"] for r in results])
    expansion_labels = np.array([r["expansion_label"] for r in results])
    oiq_labels = np.array([r["oiq_label"] for r in results])

    clean_probs = np.array([r["clean_prob"] for r in results])
    expansion_probs = np.array([r["expansion_prob"] for r in results])
    oiq_probs = np.array([r["oiq_prob"] for r in results])

    clean_evs = np.array([r["clean_ev"] for r in results])
    expansion_evs = np.array([r["expansion_ev"] for r in results])
    oiq_evs = np.array([r["oiq_ev"] for r in results])

    # Compute metrics (retro skipped)
    return compute_metrics(
        clean_labels,
        expansion_labels,
        oiq_labels,
        clean_probs,
        expansion_probs,
        oiq_probs,
        clean_evs,
        expansion_evs,
        oiq_evs,
        thresholds,
    )


def apply_thresholds(value, thresholds):
    """Map expected value to a discrete class prediction."""
    thresholds = sorted(thresholds)
    pred = 0
    for t in thresholds:
        if value > t:
            pred += 1
    return pred


def visualize_image_grid(model, dataloader, thresholds, n=24, device="cuda"):
    """
    Visualize the first three batches with a top-left anchored overlay
    and true label captions below each image.
    """
    model.eval()
    model.to(device)

    # Normalization parameters
    img_mean = [0.64041256, 0.36125767, 0.31330117]
    img_std = [0.18983584, 0.15554344, 0.14093774]

    def denormalize_image(tensor, mean, std):
        mean = torch.tensor(mean).view(3, 1, 1)
        std = torch.tensor(std).view(3, 1, 1)
        img = tensor * std + mean
        return img.clamp(0, 1)

    batch_list = []
    for i, batch in enumerate(dataloader):
        batch_list.append(batch)
        if i == 10:
            break

    # Concatenate
    images = torch.cat([b[0] for b in batch_list], dim=0).to(device)
    mucosal_cleaning_label = torch.cat([b[1] for b in batch_list], dim=0)
    expansion_label = torch.cat([b[2] for b in batch_list], dim=0)
    oiq_label = torch.cat([b[3] for b in batch_list], dim=0)

    with torch.no_grad():
        y_clean, y_expansion, y_oiq, y_retro = model(images)
        clean_prob = torch.softmax(y_clean, dim=1)
        expansion_prob = torch.softmax(y_expansion, dim=1)
        oiq_prob = torch.softmax(y_oiq, dim=1)

        class_indices = torch.arange(3, device=device).float()
        clean_ev = (clean_prob * class_indices).sum(dim=1)
        expansion_ev = (expansion_prob * class_indices).sum(dim=1)
        oiq_ev = (oiq_prob * class_indices).sum(dim=1)

    label_text = {0: "Poor", 1: "Adequate", 2: "Good"}

    n = min(n, images.size(0))
    plt.figure(figsize=(16, 4 * (n // 4 + 1)), dpi=100)

    for i in range(n):
        img = denormalize_image(images[i].cpu(), img_mean, img_std)
        img = img.permute(1, 2, 0).numpy()

        ax = plt.subplot(int(np.ceil(n / 4)), 4, i + 1)
        ax.imshow(img)
        ax.axis("off")

        vals = {
            "OIQ": oiq_ev[i].item(),
            "Expansion": expansion_ev[i].item(),
            "Cleaning": clean_ev[i].item(),
        }

        preds = {
            "OIQ": apply_thresholds(vals["OIQ"], thresholds["oiq"]),
            "Expansion": apply_thresholds(vals["Expansion"], thresholds["expansion"]),
            "Cleaning": apply_thresholds(vals["Cleaning"], thresholds["clean"]),
        }

        color_map = {0: "#D0021B", 1: "#F8E71C", 2: "#7ED321"}

        # Layout parameters
        bar_width = 0.35
        bar_height = 0.03
        label_to_bar_gap = 0.05
        bar_group_gap = 0.03
        left_padding = 0.02
        top_padding = 0.025
        bottom_padding = 0.02
        font_size = 12

        total_height = top_padding + bottom_padding + len(vals) * (bar_height + label_to_bar_gap + bar_group_gap)
        total_width = left_padding + bar_width + 0.04

        # Overlay box
        ax.add_patch(
            Rectangle(
                (0.0, 1.0 - total_height),
                total_width,
                total_height,
                transform=ax.transAxes,
                color=(0, 0, 0, 0.55),
                linewidth=0,
                zorder=2,
            )
        )

        current_y = 1.0 - top_padding
        for k, v in vals.items():
            pred = preds[k]
            color = color_map[pred]

            ax.text(
                left_padding,
                current_y,
                f"{k}",
                transform=ax.transAxes,
                fontsize=font_size,
                fontweight="bold",
                color="white",
                va="top",
                ha="left",
                zorder=5,
            )

            current_y -= label_to_bar_gap + bar_height
            ax.add_patch(
                Rectangle(
                    (left_padding, current_y),
                    bar_width,
                    bar_height,
                    transform=ax.transAxes,
                    color=(0.25, 0.25, 0.25, 0.85),
                    linewidth=0,
                    zorder=3,
                )
            )

            fill_width = (v / 2.0) * bar_width
            ax.add_patch(
                Rectangle(
                    (left_padding, current_y),
                    fill_width,
                    bar_height,
                    transform=ax.transAxes,
                    color=color,
                    linewidth=0,
                    zorder=4,
                )
            )

            current_y -= bar_group_gap

        # Add caption with true labels below image
        oiq_lbl = label_text.get(int(oiq_label[i].item()), "Unknown")
        exp_lbl = label_text.get(int(expansion_label[i].item()), "Unknown")
        clean_lbl = label_text.get(int(mucosal_cleaning_label[i].item()), "Unknown")

        caption = f"OIQ: {oiq_lbl} | Exp: {exp_lbl} | Clean: {clean_lbl} | idx: {i}"
        ax.set_title(caption, fontsize=11, color="black", pad=8)

    plt.tight_layout()
    plt.show()


def visualize_image_single(
    model,
    dataloader,
    thresholds,
    n=24,
    device="cuda",
    save_index=None,
    save_dir="ev_visuals",
):
    """
    Visualize the first three batches with EV bars. Optionally save a single image
    if `save_index` is provided. Saved image names are based on TRUE LABELS.

    Args:
        model: Trained PyTorch model.
        dataloader: PyTorch DataLoader.
        thresholds: Dict of thresholds per task.
        n: Max number of images to visualize.
        device: "cuda" or "cpu".
        save_index: If set (int), saves only that image's visualization.
        save_dir: Folder to save image to.
    """
    model.eval()
    model.to(device)

    os.makedirs(save_dir, exist_ok=True)

    # Normalization values
    img_mean = [0.64041256, 0.36125767, 0.31330117]
    img_std = [0.18983584, 0.15554344, 0.14093774]

    def denormalize_image(tensor, mean, std):
        mean = torch.tensor(mean).view(3, 1, 1)
        std = torch.tensor(std).view(3, 1, 1)
        img = tensor * std + mean
        return img.clamp(0, 1)

    # Collect first 3 batches
    batch_list = []
    for i, batch in enumerate(dataloader):
        batch_list.append(batch)
        if i == 10:
            break

    # Concatenate them
    images = torch.cat([b[0] for b in batch_list], dim=0).to(device)
    mucosal_cleaning_label = torch.cat([b[1] for b in batch_list], dim=0)
    expansion_label = torch.cat([b[2] for b in batch_list], dim=0)
    oiq_label = torch.cat([b[3] for b in batch_list], dim=0)

    with torch.no_grad():
        y_clean, y_expansion, y_oiq, y_retro = model(images)
        clean_prob = torch.softmax(y_clean, dim=1)
        expansion_prob = torch.softmax(y_expansion, dim=1)
        oiq_prob = torch.softmax(y_oiq, dim=1)

        class_indices = torch.arange(3, device=device).float()
        clean_ev = (clean_prob * class_indices).sum(dim=1)
        expansion_ev = (expansion_prob * class_indices).sum(dim=1)
        oiq_ev = (oiq_prob * class_indices).sum(dim=1)

    # Category name mapping
    label_text = {0: "Poor", 1: "Adequate", 2: "Good"}

    n = min(n, images.size(0))
    plt.figure(figsize=(16, 4 * (n // 4 + 1)), dpi=100)

    for i in range(n):
        img = denormalize_image(images[i].cpu(), img_mean, img_std)
        img_np = img.permute(1, 2, 0).numpy()

        vals = {
            "OIQ": oiq_ev[i].item(),
            "Expansion": expansion_ev[i].item(),
            "Cleaning": clean_ev[i].item(),
        }

        preds = {
            "OIQ": apply_thresholds(vals["OIQ"], thresholds["oiq"]),
            "Expansion": apply_thresholds(vals["Expansion"], thresholds["expansion"]),
            "Cleaning": apply_thresholds(vals["Cleaning"], thresholds["clean"]),
        }

        color_map = {0: "#D0021B", 1: "#F8E71C", 2: "#7ED321"}

        # Layout parameters
        bar_width = 0.35
        bar_height = 0.03
        label_to_bar_gap = 0.05
        bar_group_gap = 0.03
        left_padding = 0.02
        top_padding = 0.025
        bottom_padding = 0.02
        font_size = 12

        total_height = top_padding + bottom_padding + len(vals) * (bar_height + label_to_bar_gap + bar_group_gap)
        total_width = left_padding + bar_width + 0.04

        fig, ax = plt.subplots(figsize=(4, 4), dpi=150)
        ax.imshow(img_np)
        ax.axis("off")

        # Overlay box
        ax.add_patch(
            Rectangle(
                (0.0, 1.0 - total_height),
                total_width,
                total_height,
                transform=ax.transAxes,
                color=(0, 0, 0, 0.55),
                linewidth=0,
                zorder=2,
            )
        )

        current_y = 1.0 - top_padding
        for k, v in vals.items():
            pred = preds[k]
            color = color_map[pred]

            # Label
            ax.text(
                left_padding,
                current_y,
                f"{k}",
                transform=ax.transAxes,
                fontsize=font_size,
                fontweight="bold",
                color="white",
                va="top",
                ha="left",
                zorder=5,
            )

            # Bar below label
            current_y -= label_to_bar_gap + bar_height
            ax.add_patch(
                Rectangle(
                    (left_padding, current_y),
                    bar_width,
                    bar_height,
                    transform=ax.transAxes,
                    color=(0.25, 0.25, 0.25, 0.85),
                    linewidth=0,
                    zorder=3,
                )
            )

            # Filled portion
            fill_width = (v / 2.0) * bar_width
            ax.add_patch(
                Rectangle(
                    (left_padding, current_y),
                    fill_width,
                    bar_height,
                    transform=ax.transAxes,
                    color=color,
                    linewidth=0,
                    zorder=4,
                )
            )

            current_y -= bar_group_gap

        # Save only if this index is chosen
        if save_index is not None and i == save_index:
            oiq_lbl = oiq_label[i].item()
            expansion_lbl = expansion_label[i].item()
            cleaning_lbl = mucosal_cleaning_label[i].item()

            name = (
                f"OIQ_{label_text.get(oiq_lbl, 'Unknown')}_"
                f"Expansion_{label_text.get(expansion_lbl, 'Unknown')}_"
                f"Cleaning_{label_text.get(cleaning_lbl, 'Unknown')}.png"
            )
            save_path = os.path.join(save_dir, name)
            plt.savefig(save_path, bbox_inches="tight", pad_inches=0, dpi=600)

        plt.close(fig) if save_index is not None else plt.show()


def plot_confusion_matrix(
    task_results,
    category_name,
    class_names=None,
    normalize="true",
    cmap="Blues",
    hide_y_labels=False,
    save_path=None,  # <--- directory path or full file path
    save_name=None,  # <--- optional custom file name
):
    """
    Computes and plots a labeled confusion matrix with larger tick and axis label fonts.
    When hide_y_labels=True, hides the y-axis tick labels and title.
    When save_path is provided, saves the figure to that path (with save_name if given).
    """
    labels = np.array(task_results["labels"])
    preds = np.array(task_results["preds"])

    cm = confusion_matrix(labels, preds, labels=np.unique(labels), normalize=normalize)
    fmt = ".2f" if normalize else "d"

    plt.figure(figsize=(4, 4))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        cbar=False,
        xticklabels=class_names if class_names else np.unique(labels),
        yticklabels=class_names if class_names else np.unique(labels),
        annot_kws={"size": 11},  # numbers inside boxes
    )

    # Axis labels
    ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=12, fontweight="bold")

    # Tick label sizes
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # Optionally hide y-axis labels and title
    if hide_y_labels:
        ax.set_ylabel("")  # Remove y-axis title
        ax.set_yticklabels([])  # Remove tick labels
        ax.tick_params(axis="y", length=0)  # Remove tick marks

    plt.tight_layout()

    if save_path is not None:
        os.makedirs(save_path, exist_ok=True)
        if save_name is None:
            save_name = f"{category_name}_confusion_matrix.png"
        save_full_path = os.path.join(save_path, save_name)

        plt.savefig(save_full_path, dpi=300, bbox_inches="tight")

    plt.show()


def get_expected_values(
    model,
    dataloader,
    device="cuda",
    thresholds_set="frozen",  # can be "frozen" or "finetuned"
):
    """
    Runs the model on the dataloader and computes expected values and predicted labels
    for Cleaning, Expansion, and OIQ tasks, while masking out labels == -100.
    """
    frozen_thresholds = {
        "clean": [0.640, 1.552],
        "expansion": [0.506, 1.508],
        "oiq": [0.672, 1.482],
    }

    finetuned_thresholds = {
        "clean": [0.462, 1.542],
        "expansion": [0.532, 1.580],
        "oiq": [0.572, 1.560],
    }

    thresholds = frozen_thresholds if thresholds_set == "frozen" else finetuned_thresholds

    model.eval()
    model.to(device)

    results = {
        "clean": {"evs": [], "labels": [], "preds": []},
        "expansion": {"evs": [], "labels": [], "preds": []},
        "oiq": {"evs": [], "labels": [], "preds": []},
    }

    with torch.no_grad():
        for batch in dataloader:
            images, clean_labels, expansion_labels, oiq_labels, retro_labels = batch
            images = images.to(device)

            # Forward pass
            y_clean, y_expansion, y_oiq, y_retro = model(images)

            # Convert logits to probabilities
            clean_prob = torch.softmax(y_clean, dim=1)
            expansion_prob = torch.softmax(y_expansion, dim=1)
            oiq_prob = torch.softmax(y_oiq, dim=1)

            # Expected values
            class_indices = torch.arange(3, device=device).float()
            clean_ev = (clean_prob * class_indices).sum(dim=1).cpu().numpy()
            expansion_ev = (expansion_prob * class_indices).sum(dim=1).cpu().numpy()
            oiq_ev = (oiq_prob * class_indices).sum(dim=1).cpu().numpy()

            # Thresholding function
            def apply_thresholds(ev_array, ths):
                if len(ths) == 2:
                    return np.digitize(ev_array, bins=ths).astype(int)
                elif len(ths) == 1:
                    return (ev_array >= ths[0]).astype(int)
                else:
                    raise ValueError("Threshold list must contain 1 or 2 values.")

            clean_preds = apply_thresholds(clean_ev, thresholds["clean"])
            expansion_preds = apply_thresholds(expansion_ev, thresholds["expansion"])
            oiq_preds = apply_thresholds(oiq_ev, thresholds["oiq"])

            # Convert labels to numpy
            clean_labels_np = clean_labels.cpu().numpy()
            expansion_labels_np = expansion_labels.cpu().numpy()
            oiq_labels_np = oiq_labels.cpu().numpy()

            # Store only valid entries
            results["clean"]["evs"].extend(clean_ev)
            results["clean"]["labels"].extend(clean_labels_np)
            results["clean"]["preds"].extend(clean_preds)

            results["expansion"]["evs"].extend(expansion_ev)
            results["expansion"]["labels"].extend(expansion_labels_np)
            results["expansion"]["preds"].extend(expansion_preds)

            results["oiq"]["evs"].extend(oiq_ev)
            results["oiq"]["labels"].extend(oiq_labels_np)
            results["oiq"]["preds"].extend(oiq_preds)

    return results


def plot_expected_value_distributions(task_results, task_name="Unknown"):
    """
    Plots smoothed distributions (KDEs) of expected values per true class label
    for a single task (e.g., results["oiq"]).

    Args:
        task_results (dict): Dictionary with keys "evs" and "labels" for one task.
        task_name (str): Name of the task (e.g., "clean", "expansion", "oiq").
    """
    class_names = {0: "Poor", 1: "Adequate", 2: "Good"}
    colors = {0: "red", 1: "orange", 2: "green"}

    evs = np.array(task_results["evs"])
    labels = np.array(task_results["labels"])

    plt.figure(figsize=(6, 4))
    for c in [0, 1, 2]:
        mask = labels == c
        if np.any(mask):
            sns.kdeplot(
                evs[mask],
                label=class_names[c],
                color=colors[c],
                fill=False,
                linewidth=2,
            )

    plt.title(f"Expected Value Distribution - {task_name.capitalize()}")
    plt.xlabel("Expected Value")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()


def metrics_to_dataframe(metrics_dict):
    rows = []
    for task, metrics in metrics_dict.items():
        row = {
            "task": task,
            "AUROC": metrics["auroc_macro"],
            "BA": metrics["balanced_acc_macro"],
            "Macro MAE": metrics["mae_macro"],
        }
        # add per-class F1 scores with explicit column names
        for i, recall in enumerate(metrics["balanced_acc_per_class"]):
            row[f"recall_class_{i}"] = recall
        rows.append(row)
    return pd.DataFrame(rows).set_index("task")

#### Load models

In [ ]:
from src.models.heads import MlpHead
from src.models.backbones import CaformerBackbone

in_features = 64 + 128 + 320 + 512


class FrozenBackboneCADq(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CaformerBackbone(weights_path=parent_dir / "model_weights/backbone/ca_s18_finetuned.pth")
        self.cadq_head = MlpHead(
            in_features=in_features, weights_path=parent_dir / "model_weights/head/supervised_frozen_mlp_head.pth"
        )

    def forward(self, x):
        cls, features = self.backbone(x)
        y_clean, y_expansion, y_oiq, y_retro = self.cadq_head(features)
        return y_clean, y_expansion, y_oiq, y_retro


class FinetunedBackboneCADq(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        self.backbone = CaformerBackbone()
        self.head = MlpHead(in_features=in_features)

        if weights_path:
            weights_path = Path(weights_path)
            if not weights_path.is_file():
                error_message = f"Pretrained weights not found at {weights_path}"
                raise FileNotFoundError(error_message)

            state_dict = torch.load(weights_path, map_location="cpu")
            self.load_state_dict(state_dict, strict=True)
        else:
            error_message = "No pretrained weights provided for the model."
            raise ValueError(error_message)

    def forward(self, x):
        cls, features = self.backbone(x)
        y_clean, y_expansion, y_oiq, y_retro = self.head(features)
        return y_clean, y_expansion, y_oiq, y_retro


frozen_model = FrozenBackboneCADq()
finetuned_model = FinetunedBackboneCADq(weights_path=parent_dir / "model_weights/supervised_finetuned_cadq_mlp.pth")

#### Define thresholds

In [ ]:
frozen_thresholds = {
    "clean": [0.640, 1.552],
    "expansion": [0.506, 1.508],
    "oiq": [0.672, 1.482],
    "retro": [0.510],
}

finetuned_thresholds = {
    "clean": [0.462, 1.542],
    "expansion": [0.532, 1.580],
    "oiq": [0.572, 1.560],
    "retro": [0.685],
}

#### Load test data

In [ ]:
dataframe_test = load_image_dataframe(data_path=parent_dir / "data/iqa", split="test")

test_dataset = ImageDataset(
    dataframe=dataframe_test,
    transform=get_transforms(train=False),
    preprocess=True,
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

##### Visualize images

In [ ]:
visualize_image_grid(frozen_model, test_loader, frozen_thresholds, n=8, device="cuda")

In [ ]:
visualize_image_single(frozen_model, test_loader, frozen_thresholds, n=200, device="cuda", save_index=168)

#### Compute metrics

In [ ]:
eval_frozen = evaluate_model(frozen_model, test_loader, frozen_thresholds)
df = metrics_to_dataframe(eval_frozen).round(3)
print(df)

In [ ]:
eval_finetuned = evaluate_model(finetuned_model, test_loader, finetuned_thresholds)
df = metrics_to_dataframe(eval_finetuned).round(3)
print(df)

#### Plot smoothed density plots

In [ ]:
results_frozen = get_expected_values(frozen_model, test_loader, device="cuda", thresholds_set="frozen")
results_finetuned = get_expected_values(finetuned_model, test_loader, device="cuda", thresholds_set="finetuned")

In [ ]:
plot_expected_value_distributions(results_frozen["clean"], task_name="clean")
plot_expected_value_distributions(results_frozen["expansion"], task_name="expansion")
plot_expected_value_distributions(results_frozen["oiq"], task_name="oiq")

plot_expected_value_distributions(results_finetuned["clean"], task_name="clean")
plot_expected_value_distributions(results_finetuned["expansion"], task_name="expansion")
plot_expected_value_distributions(results_finetuned["oiq"], task_name="oiq")

#### Plot confusion matrices

In [ ]:
plot_confusion_matrix(
    results_frozen["oiq"],
    category_name="oiq",
    class_names=["Poor", "Adequate", "Good"],
    hide_y_labels=False,
    save_name="oiq_confusion_matrix_frozen.png",
    save_path="results",
)
plot_confusion_matrix(
    results_finetuned["oiq"],
    category_name="oiq",
    class_names=["Poor", "Adequate", "Good"],
    hide_y_labels=False,
    save_name="oiq_confusion_matrix_finetuned.png",
    save_path="results",
)
plot_confusion_matrix(
    results_frozen["expansion"],
    category_name="expansion",
    class_names=["Poor", "Adequate", "Good"],
    hide_y_labels=False,
    save_name="expansion_confusion_matrix_frozen.png",
    save_path="results",
)
plot_confusion_matrix(
    results_finetuned["expansion"],
    category_name="expansion",
    class_names=["Poor", "Adequate", "Good"],
    hide_y_labels=False,
    save_name="expansion_confusion_matrix_finetuned.png",
    save_path="results",
)
plot_confusion_matrix(
    results_frozen["clean"],
    category_name="clean",
    class_names=["Poor", "Adequate", "Good"],
    hide_y_labels=False,
    save_name="clean_confusion_matrix_frozen.png",
    save_path="results",
)
plot_confusion_matrix(
    results_finetuned["clean"],
    category_name="clean",
    class_names=["Poor", "Adequate", "Good"],
    hide_y_labels=False,
    save_name="clean_confusion_matrix_finetuned.png",
    save_path="results",
)